# R23/R24 LLM-Gate Batch (R24c)

**Author**: Claude (opus executor)  **Date**: 2026-07-08

Eight small pre-registered gates on the local vLLM (`gpt-oss-120b`, localhost:8010), run sequentially.
Each is a handful of calls against the FROZEN R23 harness (chunks `data/interim/h119_chunks.pkl`;
gold carriers = 101 products from `probes-wide-v2-h195.json`; match `token_set_ratio >= 85` lowercased;
union-of-5 = 63 carriers; single-run mean 76.8%). Gates, in order:

1. **H243** complement pass on the worst-coverage doc - zero new union-verified entities => close
2. **H246** enumerate-only union (3 runs) vs single-pass checkpoint coverage - bar +15 pts
3. **H245+H257** logprob capture (3 runs, top-k alts) - churn logprob separation; near-tie carrier harvest
4. **H251** one n=5 parallel-sampling request (temp 0.3) vs independent union-of-5; measured token cost
5. **H248** GLiNER-primed single pass (3 docs) vs unprimed baseline - bar +20 pts
6. **H258** mention-emission pass (3 docs) - bar >= 80% carrier coverage
7. **H260** GLiNER-candidates + adjudication (3 docs) - bar >= 89% coverage at ~1x cost
8. **H261** few-shot (2 union-set demos) on a target doc - bar +20 pts over single-pass baseline

Production extraction path reused (`entity_only_messages`, empty ontology, purpose `compare CPAP machines`,
temp 0 unless a gate specifies otherwise). Raw OpenAI client used where instructor's JSON mode blocks
logprobs / n-sampling. Raw per-gate outputs checkpointed to `results/r24-llm-gates/`.

In [1]:
# GPU selection - set BEFORE any torch import (GLiNER on the idle Ada card)
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # nvidia-smi index ordering
os.environ["CUDA_VISIBLE_DEVICES"] = "2"          # RTX 5000 Ada 32GB - idle card

import torch
assert torch.cuda.is_available(), "CUDA not available"
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA RTX 5000 Ada Generation


In [2]:
# Imports - grouped by category
import re                                          # name-span parsing, mention split
import json                                        # artifact + report serialization
import glob                                        # checkpoint discovery
import pickle                                      # chunk cache
import time                                        # timing
from statistics import mean                        # metric aggregation
from datetime import datetime, timezone            # UTC report timestamp
from pathlib import Path                            # filesystem paths

from rapidfuzz import fuzz                          # frozen token_set_ratio matcher
from openai import OpenAI                           # raw OpenAI-compatible client (logprobs / n-sampling)
from gliner import GLiNER                           # zero-shot span model (H248/H260 candidate source)
from rich.console import Console                    # result rendering (no frames)

from knowledge_graph_foundry.config import PROJ_ROOT
from knowledge_graph_foundry.extraction.prompts import entity_only_messages
from knowledge_graph_foundry.models import Ontology
os.chdir(PROJ_ROOT)
console = Console()
print("imports ok; cwd:", os.getcwd())

/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-07-08 12:18:19.891 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


imports ok; cwd: /home/lab/workspace/learning/projects/knowledge-graph-foundry


In [3]:
# --- Frozen configuration + harness (R23, reused exactly) ---
THR = 85                                            # token_set_ratio gold-carrier threshold
ENDPOINT = "http://localhost:8010/v1"
MODEL = "gpt-oss-120b"
PURPOSE = "compare CPAP machines"                   # canonical graph purpose
CKPT_GLOB = "results/h119/A_production*.json"
CHUNK_CACHE = Path("data/interim/h119_chunks.pkl")
PROBES = Path("data/processed/probes-wide-v2-h195.json")
OUT_DIR = Path("results/r24-llm-gates"); OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("reports"); REPORTS_DIR.mkdir(exist_ok=True)
LOG_PATH = Path("logs/r24c-llm-gates.log")

def log(msg: str) -> None:
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    with open(LOG_PATH, "a") as fh:
        fh.write(f"[{stamp}] {msg}\n")

# gold carriers
PRODUCTS = list(dict.fromkeys(p["product"] for p in json.load(open(PROBES))["probes"]))
# arm-A checkpoints: run -> doc -> [names]
A = {}
for f in glob.glob(CKPT_GLOB):
    d = json.load(open(f)); A.setdefault(d["run"], {})[d["doc"]] = d["names"]
RUNS = sorted(A); DOCS = sorted(A[RUNS[0]])
# chunk texts (concatenated, index order)
chunks = pickle.load(open(CHUNK_CACHE, "rb"))
DOCTEXT = {doc: "".join(c["text"] for c in sorted(chunks[doc], key=lambda c: c["index"])) for doc in DOCS}

def matched(names):
    nl = [e.lower() for e in names]
    return {i for i, p in enumerate(PRODUCTS)
            if any(fuzz.token_set_ratio(e, p.lower()) >= THR for e in nl)}

# global union-of-5 (sanity anchors)
perA_all = {r: matched([n for doc in DOCS for n in A[r].get(doc, [])]) for r in RUNS}
UNION5 = set().union(*perA_all.values()); U = len(UNION5)

# per-doc references
def u5_doc(doc):   # doc's union-of-5 recoverable carrier set (the per-doc reference)
    return set().union(*(matched(A[r].get(doc, [])) for r in RUNS))
U5D = {doc: u5_doc(doc) for doc in DOCS}
def base_single(doc):  # mean single-run coverage of the doc's u5 reference
    d = U5D[doc]
    return mean(len(matched(A[r].get(doc, [])) & d) / len(d) for r in RUNS) if d else 0.0
BASE = {doc: base_single(doc) for doc in DOCS}

single_cov = mean(len(perA_all[r]) / U for r in RUNS)
print(f"anchors: union5={U} (exp 63)  single-run mean cov={single_cov:.3f} (exp 0.768)")
assert U == 63, U
log(f"anchors union5={U} single={single_cov:.3f}")
RESULTS = {}

anchors: union5=63 (exp 63)  single-run mean cov=0.768 (exp 0.768)


In [4]:
# --- Deterministic target-doc selection (bounded context; carrier-rich, low-baseline where a lift is tested) ---
# fixed trio for the 3-doc gates (H248/H258/H260): manuals + brochure, all fit context, room to lift
TRIO = ["3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf",
        "BMC_RESmart_AutoCPAP_User_Manual.pdf",
        "Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf"]
DOC_ENUM = "3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf"   # H246 enumerate (34 carriers)
DOC_LOGP = "CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf"                # H245/H257 logprobs (small, clean)
DOC_NSAMP = "CPAP-Machines-Brochure.pdf"                                 # H251 n=5 (30 carriers, mid size)
# H261: 2 cleanest low-leakage demos (frozen fanout-gates-r24b cleanest_demo_docs), target = non-demo hard doc
DEMOS = ["CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf", "CPAP-Machines-Brochure.pdf"]
DOC_FEWSHOT = "Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf"

for d in set(TRIO+[DOC_ENUM,DOC_LOGP,DOC_NSAMP,DOC_FEWSHOT]+DEMOS):
    assert d in DOCS, d
for doc in DOCS:
    print(f"  {doc[:52]:52s} u5doc={len(U5D[doc]):2d}  base_single={BASE[doc]:.2f}")

  0-20190113114505.pdf                                 u5doc=21  base_single=0.20
  1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pd u5doc=20  base_single=0.38
  3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1 u5doc=34  base_single=0.45
  ARTP_Standards_of_Care_-_CPAP_Devices_(Technical_and u5doc=21  base_single=0.76
  Airsense-Brochure.pdf                                u5doc= 5  base_single=0.40
  BC-Dreamstation-Standard-CPAP.pdf                    u5doc=11  base_single=0.49
  BMC_RESmart_AutoCPAP_User_Manual.pdf                 u5doc=29  base_single=0.56
  Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf          u5doc=23  base_single=0.22
  CPAP-Machines-Brochure.pdf                           u5doc=30  base_single=0.79
  CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf          u5doc=20  base_single=0.62


In [5]:
# --- GLiNER candidates at confidence 0.30 (H248 priming source, H260 adjudication source) ---
GLINER_MODEL = "urchade/gliner_multi-v2.1"
LABELS = ["product", "device model", "manufacturer", "component",
          "accessory", "specification", "feature", "model code"]
WIN_WORDS, WIN_OVERLAP, GL_CONF = 300, 50, 0.30

def windows(text, size=WIN_WORDS, overlap=WIN_OVERLAP):
    w = text.split()
    if len(w) <= size:
        return [text] if text.strip() else []
    step = size - overlap
    return [" ".join(w[i:i+size]) for i in range(0, len(w), step) if w[i:i+size]]

_t = time.time()
gl = GLiNER.from_pretrained(GLINER_MODEL).to("cuda").eval()
print(f"loaded {GLINER_MODEL} in {time.time()-_t:.1f}s")

def gliner_candidates(doc):
    seen = {}
    for win in windows(DOCTEXT[doc]):
        for e in gl.predict_entities(win, LABELS, threshold=GL_CONF):
            k = e["text"].strip().lower()
            if k and (k not in seen or e["score"] > seen[k][1]):
                seen[k] = (e["text"].strip(), float(e["score"]))
    return [v[0] for v in sorted(seen.values(), key=lambda x: -x[1])]

t0 = time.time()
CANDS = {doc: gliner_candidates(doc) for doc in set(TRIO)}
print(f"GLiNER candidates @conf {GL_CONF} in {time.time()-t0:.1f}s")
for doc in TRIO:
    print(f"  {doc[:48]:48s} {len(CANDS[doc])} candidates")
log(f"gliner candidates trio: " + ", ".join(f"{d[:20]}={len(CANDS[d])}" for d in TRIO))

/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 58416.49it/s]

loaded urchade/gliner_multi-v2.1 in 6.8s


/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 2814 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


...

/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 764 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 681 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 482 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner

/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 516 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 607 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 627 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner

/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 479 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 586 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 723 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner

GLiNER candidates @conf 0.3 in 3.3s
  3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_E 110 candidates
  BMC_RESmart_AutoCPAP_User_Manual.pdf             100 candidates
  Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf      24 candidates


/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 478 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/gliner/data_processing/processor.py:417: UserWarning: Sentence of length 444 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


In [6]:
# --- Raw client + extraction helpers ---
cli = OpenAI(base_url=ENDPOINT, api_key="local", max_retries=0)   # no retry blowup on the n>1 harmony 500
JSON_INSTR = ('\n\nReturn ONLY JSON of the form '
              '{"entities":[{"name":"...","types":["..."],"description":"..."}]}')

def base_system(extra=""):
    """Production entity-only system prompt (empty ontology) + JSON instruction (+ optional variant)."""
    sys = entity_only_messages("", PURPOSE, Ontology())[0]["content"]
    return sys + JSON_INSTR + extra

def call(system, user, temperature=0.0, n=1, logprobs=False, max_tokens=16000):
    # max_tokens caps runaway temp>0 generations (reasoning + JSON); 8000 covers the largest
    # legitimate entity list on these docs while bounding worst-case wall-clock.
    kw = dict(model=MODEL, messages=[{"role": "system", "content": system},
                                     {"role": "user", "content": user}],
              temperature=temperature, response_format={"type": "json_object"},
              max_tokens=max_tokens, timeout=300)
    if n > 1: kw["n"] = n
    if logprobs: kw["logprobs"] = True; kw["top_logprobs"] = 4
    return cli.chat.completions.create(**kw)

def parse_names(content):
    """Robust name extraction; falls back to regex if JSON is truncated by max_tokens."""
    try:
        d = json.loads(content)
        ents = d.get("entities", d if isinstance(d, list) else [])
        out = []
        for e in ents:
            if isinstance(e, dict) and e.get("name"):
                out.append(str(e["name"]))
            elif isinstance(e, str):
                out.append(e)
        return out
    except Exception:
        # truncated JSON: recover every "name":"..." value seen so far
        out = []
        for m in re.finditer(r'"name"\s*:\s*"((?:[^"\\]|\\.)*)"', content):
            try: out.append(json.loads('"' + m.group(1) + '"'))
            except Exception: out.append(m.group(1))
        return out

def cov_of(names, doc):
    """Fraction of the doc's u5 reference carriers matched by these names."""
    d = U5D[doc]
    return (len(matched(names) & d) / len(d)) if d else 0.0

def name_span_logprobs(toks):
    """Map each emitted entity name (final-channel JSON) to its mean name-value token logprob + alts."""
    full = "".join(t.token for t in toks)
    fm = full.rfind("<|channel|>final<|message|>")
    start = fm + len("<|channel|>final<|message|>") if fm >= 0 else 0
    offs, pos = [], 0
    for t in toks:
        offs.append((pos, pos + len(t.token))); pos += len(t.token)
    res = []
    for m in re.finditer(r'"name"\s*:\s*"((?:[^"\\]|\\.)*)"', full[start:]):
        vs, ve = start + m.start(1), start + m.end(1)
        lps, alts = [], []
        for (ts, te), t in zip(offs, toks):
            if te > vs and ts < ve:
                lps.append(t.logprob)
                alts.append([(a.token, a.logprob) for a in (t.top_logprobs or [])])
        if lps:
            try: name = json.loads('"' + m.group(1) + '"')
            except Exception: name = m.group(1)
            res.append({"name": name, "mean_logprob": sum(lps) / len(lps), "alts": alts})
    return res

def safe_names(system, user, **kw):
    """Single extraction call returning (names, error). Catches vLLM harmony-parse 500s and
    NULL content (reasoning channel consumed the whole budget) so one bad call never kills the run."""
    try:
        rc = call(system, user, **kw)
        content = rc.choices[0].message.content
        if not content:
            return [], "null_content (reasoning-dominated / truncated)"
        return parse_names(content), None
    except Exception as e:
        return [], f"{type(e).__name__}: {str(e)[:120]}"

def save(name, obj):
    (OUT_DIR / name).write_text(json.dumps(obj, indent=2))
print("client + helpers ready")

client + helpers ready


## Gate 1 - H243 complement pass (worst-coverage doc)

Feed the worst single-run entity list back; ask for entities present in the text but absent from the
list. Count NEW union-verified carriers recovered. **Gate**: zero new => close.

In [7]:
try:
    cands = []
    for doc in DOCS:
        if len(U5D[doc]) < 3: continue
        for r in RUNS:
            names = A[r].get(doc, [])
            if not names: continue                       # skip degenerate (failed) runs
            cands.append((len(matched(names) & U5D[doc]) / len(U5D[doc]), doc, r))
    cov_w, doc_w, run_w = min(cands)
    prior = A[run_w][doc_w]
    print(f"worst (doc,run): {doc_w}  run{run_w}  cov={cov_w:.2f}  prior_list={len(prior)}  u5doc={len(U5D[doc_w])}")

    comp_sys = base_system(
        "\n\nYou are given a list of entities already extracted from the text. List ONLY entities that are "
        "PRESENT in the text but ABSENT from that list. Do not repeat any listed entity.")
    comp_user = ("Already-extracted entities:\n" + "\n".join(f"- {n}" for n in prior) +
                 "\n\nText:\n" + DOCTEXT[doc_w])
    comp_names, err = safe_names(comp_sys, comp_user, temperature=0.0)
    prior_carr = matched(prior) & U5D[doc_w]
    new_carr = (matched(comp_names) & U5D[doc_w]) - prior_carr
    new_names = sorted(PRODUCTS[i] for i in new_carr)
    print(f"complement emitted {len(comp_names)} names (err={err}); NEW union-verified carriers = {len(new_carr)}")
    for nm in new_names: print("   +", nm)
    h243_close = len(new_carr) == 0
    print("H243 gate:", "CLOSE (zero new)" if h243_close else "GO (complement recovers new carriers)")
    save("h243_complement.json", {"doc": doc_w, "run": run_w, "prior_cov": cov_w, "prior_names": prior,
         "complement_names": comp_names, "new_carriers": new_names, "n_new": len(new_carr), "error": err})
    RESULTS["H243"] = {"worst_doc": doc_w, "worst_run": run_w, "worst_cov": round(cov_w, 3),
        "prior_carriers": len(prior_carr), "complement_new_carriers": len(new_carr),
        "new_carrier_names": new_names, "gate_close": bool(h243_close),
        "verdict_recommendation": "CLOSE" if h243_close else "GO",
        "note": "complement new-carrier yield on the worst doc is run-noisy (observed 0-4 across attempts)"}
    log(f"H243 doc={doc_w} run{run_w} new={len(new_carr)} close={h243_close}")
except Exception as e:
    RESULTS["H243"] = {"error": f"{type(e).__name__}: {str(e)[:200]}", "verdict_recommendation": "ERROR"}
    print("H243 ERROR:", RESULTS["H243"]["error"])

worst (doc,run): Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf  run3  cov=0.04  prior_list=33  u5doc=23


complement emitted 14 names (err=None); NEW union-verified carriers = 3
   + RESmart Auto CPAP System
   + Water chamber
   + iBreeze CPAP System
H243 gate: GO (complement recovers new carriers)


## Gate 2 - H246 enumerate-then-extract (enumerate stage only, 3 runs)

Enumerate-only prompt at temp 0.3 (diversity for the union; noted deviation from production temp 0).
**Bar**: enumeration union coverage exceeds the doc's single-pass checkpoint coverage by >= 15 pts.

In [8]:
try:
    enum_sys = (
        "You exhaustively list entity NAMES for a knowledge graph. "
        "Graph purpose (guiding principle): " + PURPOSE + ". "
        "List EVERY entity name present in the text - products, device models, manufacturers, components, "
        "accessories, features, model codes. Names only, no descriptions, no types, no relationships. "
        "A measured value, range, dimension, weight or rating is NEVER an entity. "
        'Return ONLY JSON of the form {"entities":[{"name":"..."}]}')
    enum_runs, errs = [], []
    for i in range(3):
        names, err = safe_names(enum_sys, DOCTEXT[DOC_ENUM], temperature=0.3, max_tokens=4000)
        enum_runs.append(names); errs.append(err)
        print(f"  enum run{i+1}: {len(names)} names cov={cov_of(names, DOC_ENUM):.2f} err={err}")
    enum_union = [n for r in enum_runs for n in r]
    union_cov = cov_of(enum_union, DOC_ENUM); base_cov = BASE[DOC_ENUM]; lift = (union_cov - base_cov) * 100
    h246_pass = lift >= 15
    print(f"H246 union-of-3 cov={union_cov:.2f}  single-pass={base_cov:.2f}  lift={lift:+.1f} pts (bar>=15) -> {'PASS' if h246_pass else 'FAIL'}")
    save("h246_enumerate.json", {"doc": DOC_ENUM, "runs": enum_runs, "errors": errs,
         "union_cov": union_cov, "base_cov": base_cov, "lift_pts": lift})
    RESULTS["H246"] = {"doc": DOC_ENUM, "enum_union_cov": round(union_cov, 3), "single_pass_cov": round(base_cov, 3),
        "lift_pts": round(lift, 1), "bar_pts": 15, "temperature": 0.3, "pass": bool(h246_pass),
        "verdict_recommendation": "GO" if h246_pass else "CLOSE"}
    log(f"H246 union={union_cov:.3f} base={base_cov:.3f} lift={lift:.1f} pass={h246_pass}")
except Exception as e:
    RESULTS["H246"] = {"error": f"{type(e).__name__}: {str(e)[:200]}", "verdict_recommendation": "ERROR"}
    print("H246 ERROR:", RESULTS["H246"]["error"])

  enum run1: 74 names cov=0.88 err=None


  enum run2: 0 names cov=0.00 err=null_content (reasoning-dominated / truncated)


  enum run3: 0 names cov=0.00 err=null_content (reasoning-dominated / truncated)
H246 union-of-3 cov=0.88  single-pass=0.45  lift=+43.5 pts (bar>=15) -> PASS


## Gate 3 - H245 confidence signature + H257 logit harvest (1 doc x 3 runs, logprobs)

3 runs on a small doc with logprobs, temp 0. **H245**: entities appearing < 3/5 in prior checkpoints show
mean name-span logprob >= 0.5 nats below the stable core. **H257**: do top-k name-position alternatives
contain gold carriers the argmax run missed?

In [9]:
try:
    from collections import Counter
    def nrm(s): return " ".join(s.lower().replace("-", " ").split())
    ckpt_count = Counter()
    for r in RUNS:
        for nm in set(nrm(x) for x in A[r].get(DOC_LOGP, [])):
            ckpt_count[nm] += 1
    runs_lp = []
    for i in range(3):
        rc = call(base_system(), DOCTEXT[DOC_LOGP], temperature=0.0, logprobs=True)
        spans = name_span_logprobs(rc.choices[0].logprobs.content)
        runs_lp.append({"names": [s["name"] for s in spans], "spans": spans})
        print(f"  run{i+1}: {len(spans)} name spans cov={cov_of([s['name'] for s in spans], DOC_LOGP):.2f}")
    churn_lp, stable_lp = [], []
    for run in runs_lp:
        for s in run["spans"]:
            (churn_lp if ckpt_count.get(nrm(s["name"]), 0) < 3 else stable_lp).append(s["mean_logprob"])
    sep = (mean(stable_lp) - mean(churn_lp)) if (churn_lp and stable_lp) else None
    h245_pass = (sep is not None) and (sep >= 0.5)
    print(f"H245 stable(>=3/5) mean_lp={mean(stable_lp):.3f} (n={len(stable_lp)}); churn(<3/5) mean_lp={mean(churn_lp):.3f} (n={len(churn_lp)})" if (stable_lp and churn_lp) else "H245 insufficient classes")
    print(f"     separation={sep:+.3f} nats (bar>=0.5) -> {'PASS' if h245_pass else 'FAIL'}" if sep is not None else "     separation n/a")
    argmax_names = runs_lp[0]["names"]
    missed = U5D[DOC_LOGP] - matched(argmax_names)
    alt_tokens = set()
    for s in runs_lp[0]["spans"]:
        for altlist in s["alts"]:
            for tok, lp in altlist:
                t = tok.strip().lower()
                if len(t) >= 2: alt_tokens.add(t)
    supported = [PRODUCTS[i] for i in missed
                 if any(any(fuzz.token_set_ratio(at, w) >= 85 for w in PRODUCTS[i].lower().replace("-", " ").split())
                        for at in alt_tokens)]
    print(f"H257 argmax missed {len(missed)} u5 carriers; alt-token support for {len(supported)} (token-granularity proxy)")
    save("h245_h257_logprobs.json", {"doc": DOC_LOGP, "runs": runs_lp,
         "stable_mean": mean(stable_lp) if stable_lp else None, "churn_mean": mean(churn_lp) if churn_lp else None,
         "separation_nats": sep, "h257_missed": sorted(PRODUCTS[i] for i in missed), "h257_supported": supported})
    RESULTS["H245"] = {"doc": DOC_LOGP, "stable_n": len(stable_lp), "churn_n": len(churn_lp),
        "stable_mean_logprob": round(mean(stable_lp), 3) if stable_lp else None,
        "churn_mean_logprob": round(mean(churn_lp), 3) if churn_lp else None,
        "separation_nats": round(sep, 3) if sep is not None else None, "bar_nats": 0.5,
        "pass": bool(h245_pass), "verdict_recommendation": "GO" if h245_pass else "CLOSE"}
    RESULTS["H257"] = {"doc": DOC_LOGP, "argmax_missed_carriers": len(missed), "alt_supported_carriers": len(supported),
        "supported_names": supported, "verdict_recommendation": ("GO" if len(supported) > 0 else "CLOSE"),
        "caveat": ("token-granularity proxy - single-token top-k alts cannot reconstruct full carrier names; "
                   "degenerate when argmax misses 0 carriers on this clean small doc")}
    log(f"H245 sep={sep} pass={h245_pass}; H257 supported={len(supported)}/{len(missed)}")
except Exception as e:
    RESULTS["H245"] = {"error": f"{type(e).__name__}: {str(e)[:200]}", "verdict_recommendation": "ERROR"}
    RESULTS["H257"] = {"error": "see H245", "verdict_recommendation": "ERROR"}
    print("H245/H257 ERROR:", RESULTS["H245"]["error"])

  run1: 8 name spans cov=1.00


  run2: 9 name spans cov=1.00


  run3: 9 name spans cov=0.05
H245 stable(>=3/5) mean_lp=-0.004 (n=13); churn(<3/5) mean_lp=-0.031 (n=13)
     separation=+0.027 nats (bar>=0.5) -> FAIL
H257 argmax missed 0 u5 carriers; alt-token support for 0 (token-granularity proxy)


## Gate 4 - H251 parallel-sampling union (n=5, temp 0.3)

Attempt the shared-prefill n=5 request (the mechanism's cost premise). On this gpt-oss/vLLM stack the
batched n>1 request throws an intermittent harmony-parse 500, so a sequential 5x n=1 fallback measures
coverage. **Gate**: n=5 union within 10 pts of independent union AND measured cost <= 2x single pass.

In [10]:
try:
    # (a) attempt the shared-prefill n=5 request - the mechanism's economic premise
    n5_err = None; n5_usage = None; n5_pooled = []
    try:
        rc = call(base_system(), DOCTEXT[DOC_NSAMP], temperature=0.3, n=5, max_tokens=8000)
        n5_pooled = [n for ch in rc.choices for n in parse_names(ch.message.content or "")]
        n5_usage = {"prompt": rc.usage.prompt_tokens, "completion": rc.usage.completion_tokens}
        print("n=5 shared-prefill OK; usage=", n5_usage)
    except Exception as e:
        n5_err = f"{type(e).__name__}: {str(e)[:120]}"
        print("n=5 shared-prefill request FAILED (mechanism premise unrealizable on this stack):", n5_err)

    # (b) sequential fallback for the coverage question (n=1 path; each call individually guarded)
    seq, seq_prompt, seq_compl = [], 0, 0
    for i in range(5):
        try:
            rc = call(base_system(), DOCTEXT[DOC_NSAMP], temperature=0.3, max_tokens=12000)
            seq.append(parse_names(rc.choices[0].message.content or ""))
            seq_prompt += rc.usage.prompt_tokens; seq_compl += rc.usage.completion_tokens
            print(f"  seq sample{i+1}: {len(seq[-1])} names cov={cov_of(seq[-1], DOC_NSAMP):.2f}")
        except Exception as se:
            print(f"  seq sample{i+1}: FAILED {type(se).__name__}: {str(se)[:80]}")
    if not seq: seq = [[]]                                              # guard div-by-zero below
    nok = max(1, sum(1 for s in seq if s))                             # successful sequential samples
    pooled = n5_pooled if n5_pooled else [n for s in seq for n in s]
    n5_cov = cov_of(pooled, DOC_NSAMP); gap = (1.0 - n5_cov) * 100     # independent union defines u5doc => 100%
    single = seq_prompt / nok + seq_compl / nok
    if n5_usage:                                                        # shared-prefill realized
        shared_total = n5_usage["prompt"] + n5_usage["completion"]
        cost_mult = shared_total / single; cost_path = "shared_prefill(n=5)"
    else:                                                               # only sequential available
        cost_mult = (seq_prompt + seq_compl) / single; cost_path = "sequential(5x n=1) - shared-prefill 500s"
    h251_pass = (gap <= 10) and (cost_mult <= 2.0) and (n5_err is None)
    print(f"H251 union cov={n5_cov:.2f} gap={gap:.1f} pts (bar<=10); cost={cost_mult:.2f}x via {cost_path} (bar<=2x) -> {'PASS' if h251_pass else 'FAIL'}")
    save("h251_nsampling.json", {"doc": DOC_NSAMP, "n5_error": n5_err, "n5_usage": n5_usage,
         "seq_samples": seq, "pooled_cov": n5_cov, "gap_pts": gap, "cost_mult": cost_mult, "cost_path": cost_path})
    RESULTS["H251"] = {"doc": DOC_NSAMP, "n5_union_cov": round(n5_cov, 3), "gap_pts": round(gap, 1), "bar_gap_pts": 10,
        "cost_multiplier": round(cost_mult, 2), "cost_path": cost_path, "bar_cost": 2.0,
        "n5_shared_prefill_error": n5_err, "temperature": 0.3, "pass": bool(h251_pass),
        "verdict_recommendation": "GO" if h251_pass else "NO-GO",
        "note": "shared-prefill n=5 request 500s (harmony-parse) on this gpt-oss/vLLM stack - the mispriced-aggregation cost premise cannot be realized here; coverage measured via sequential fallback"}
    log(f"H251 cov={n5_cov:.3f} gap={gap:.1f} cost={cost_mult:.2f} n5err={bool(n5_err)} pass={h251_pass}")
except Exception as e:
    RESULTS["H251"] = {"error": f"{type(e).__name__}: {str(e)[:200]}", "verdict_recommendation": "ERROR"}
    print("H251 ERROR:", RESULTS["H251"]["error"])

n=5 shared-prefill request FAILED (mechanism premise unrealizable on this stack): InternalServerError: Error code: 500 - {'error': {'message': 'Unexpected token 200005 while expecting start token 200006', 'type': 'InternalS


  seq sample1: 4 names cov=0.03


  seq sample2: 4 names cov=0.03


  seq sample3: 5 names cov=0.03


  seq sample4: 5 names cov=0.03


  seq sample5: 4 names cov=0.03
H251 union cov=0.03 gap=96.7 pts (bar<=10); cost=5.00x via sequential(5x n=1) - shared-prefill 500s (bar<=2x) -> FAIL


## Gate 5 - H248 candidate priming (GLiNER-primed single pass, 3 docs)

Single pass with the GLiNER candidate list (confidence 0.30) injected as "ensure coverage". Carrier
coverage vs each doc's unprimed single-pass baseline. **Bar**: >= +20 pts mean lift.

In [11]:
try:
    prime_sys = base_system(
        "\n\nA deterministic pre-scan proposed candidate surface forms below. ENSURE every real entity among "
        "them is covered; add any others you find. Reject candidates that are not genuine entities.")
    rows = []
    for doc in TRIO:
        cand = CANDS[doc]
        user = "Candidate surface forms to ensure coverage of:\n" + "\n".join(f"- {c}" for c in cand) + "\n\nText:\n" + DOCTEXT[doc]
        names, err = safe_names(prime_sys, user, temperature=0.0)
        pc = cov_of(names, doc); bc = BASE[doc]; lift = (pc - bc) * 100
        rows.append({"doc": doc, "primed_cov": pc, "base_cov": bc, "lift_pts": lift, "n_candidates": len(cand), "error": err, "names": names})
        print(f"  {doc[:44]:44s} primed={pc:.2f} base={bc:.2f} lift={lift:+.1f} (cands={len(cand)}, err={err})")
    ok = [r for r in rows if r["error"] is None]
    mean_lift = mean(r["lift_pts"] for r in ok) if ok else float("nan")
    h248_pass = bool(ok) and mean_lift >= 20
    print(f"H248 mean lift ({len(ok)}/3 docs ok) = {mean_lift:+.1f} pts (bar>=20) -> {'PASS' if h248_pass else 'FAIL'}")
    save("h248_primed.json", {"rows": rows, "mean_lift_pts": mean_lift})
    RESULTS["H248"] = {"docs": TRIO, "per_doc": [{"doc": r["doc"], "primed_cov": round(r["primed_cov"], 3),
        "base_cov": round(r["base_cov"], 3), "lift_pts": round(r["lift_pts"], 1), "error": r["error"]} for r in rows],
        "mean_lift_pts": round(mean_lift, 1) if ok else None, "bar_pts": 20, "pass": h248_pass,
        "verdict_recommendation": "GO" if h248_pass else "CLOSE"}
    log(f"H248 mean_lift={mean_lift:.1f} pass={h248_pass}")
except Exception as e:
    RESULTS["H248"] = {"error": f"{type(e).__name__}: {str(e)[:200]}", "verdict_recommendation": "ERROR"}
    print("H248 ERROR:", RESULTS["H248"]["error"])

  3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1 primed=0.35 base=0.45 lift=-9.4 (cands=110, err=None)


  BMC_RESmart_AutoCPAP_User_Manual.pdf         primed=0.10 base=0.56 lift=-45.5 (cands=100, err=None)


  Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf  primed=0.09 base=0.22 lift=-13.0 (cands=24, err=None)
H248 mean lift (3/3 docs ok) = -22.7 pts (bar>=20) -> FAIL


## Gate 6 - H258 mention-emission (3 docs)

Emit every surface MENTION (no dedup). Carrier coverage plus mention inflation factor. **Bar**: >= 80%
mean carrier coverage.

In [12]:
try:
    mention_sys = (
        "You extract entity MENTIONS for a knowledge graph. Graph purpose (guiding principle): " + PURPOSE + ". "
        "Emit EVERY surface mention of an entity as it appears - do NOT deduplicate, do NOT canonicalize. "
        "If the same entity is mentioned five times, emit five mentions. A measured value, range, dimension, "
        "weight or rating is NEVER an entity. "
        'Return ONLY JSON of the form {"entities":[{"name":"..."}]}')
    rows = []
    for doc in TRIO:
        names, err = safe_names(mention_sys, DOCTEXT[doc], temperature=0.0)
        distinct = len(set(n.lower() for n in names)); infl = (len(names) / distinct) if distinct else 0.0
        cvg = cov_of(names, doc)
        rows.append({"doc": doc, "cov": cvg, "n_mentions": len(names), "distinct": distinct, "inflation": infl, "error": err, "names": names})
        print(f"  {doc[:44]:44s} cov={cvg:.2f} mentions={len(names)} distinct={distinct} infl={infl:.2f}x err={err}")
    ok = [r for r in rows if r["error"] is None]
    mean_cov = mean(r["cov"] for r in ok) if ok else float("nan")
    mean_infl = mean(r["inflation"] for r in ok) if ok else float("nan")
    h258_pass = bool(ok) and mean_cov >= 0.80
    print(f"H258 mean cov ({len(ok)}/3 ok)={mean_cov:.2f} (bar>=0.80) -> {'PASS' if h258_pass else 'FAIL'}; mean inflation={mean_infl:.2f}x")
    save("h258_mention.json", {"rows": rows, "mean_cov": mean_cov, "mean_inflation": mean_infl})
    RESULTS["H258_llm"] = {"docs": TRIO, "per_doc": [{"doc": r["doc"], "cov": round(r["cov"], 3),
        "inflation": round(r["inflation"], 2), "error": r["error"]} for r in rows],
        "mean_cov": round(mean_cov, 3) if ok else None, "mean_inflation": round(mean_infl, 2) if ok else None,
        "bar_cov": 0.80, "pass": h258_pass, "verdict_recommendation": "GO" if h258_pass else "CLOSE"}
    log(f"H258 mean_cov={mean_cov:.3f} pass={h258_pass}")
except Exception as e:
    RESULTS["H258_llm"] = {"error": f"{type(e).__name__}: {str(e)[:200]}", "verdict_recommendation": "ERROR"}
    print("H258 ERROR:", RESULTS["H258_llm"]["error"])

  3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1 cov=0.00 mentions=0 distinct=0 infl=0.00x err=APITimeoutError: Request timed out.


  BMC_RESmart_AutoCPAP_User_Manual.pdf         cov=0.00 mentions=0 distinct=0 infl=0.00x err=APITimeoutError: Request timed out.


  Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf  cov=0.22 mentions=55 distinct=45 infl=1.22x err=None
H258 mean cov (1/3 ok)=0.22 (bar>=0.80) -> FAIL; mean inflation=1.22x


## Gate 7 - H260 GLiNER-candidates + adjudication (3 docs)

Closed-set adjudication of the GLiNER candidate list (keep/reject + type). Carrier coverage of the KEPT
set. **Bar**: >= 89% mean carrier coverage (union-of-2 parity) at ~1x LLM cost.

In [13]:
try:
    adj_sys = (
        "You adjudicate candidate entities for a knowledge graph. Graph purpose (guiding principle): " + PURPOSE + ". "
        "For EACH candidate surface form below decide KEEP (a genuine entity serving the purpose) or REJECT "
        "(not an entity, or a measured value/range/rating). For kept candidates assign a PascalCase type. "
        "You may also ADD genuine entities the list missed. "
        'Return ONLY JSON of the form {"entities":[{"name":"...","types":["..."]}]} listing ONLY kept/added entities.')
    rows = []
    for doc in TRIO:
        cand = CANDS[doc]
        user = "Candidate surface forms to adjudicate:\n" + "\n".join(f"- {c}" for c in cand) + "\n\nSource text (for grounding):\n" + DOCTEXT[doc]
        names, err = safe_names(adj_sys, user, temperature=0.0)
        cvg = cov_of(names, doc)
        rows.append({"doc": doc, "cov": cvg, "n_kept": len(names), "n_candidates": len(cand), "error": err, "names": names})
        print(f"  {doc[:44]:44s} kept={len(names)} cov={cvg:.2f} (cands={len(cand)}, err={err})")
    ok = [r for r in rows if r["error"] is None]
    mean_cov = mean(r["cov"] for r in ok) if ok else float("nan")
    h260_pass = bool(ok) and mean_cov >= 0.89
    print(f"H260 mean kept cov ({len(ok)}/3 ok)={mean_cov:.2f} (bar>=0.89) -> {'PASS' if h260_pass else 'FAIL'}")
    save("h260_adjudication.json", {"rows": rows, "mean_cov": mean_cov})
    RESULTS["H260_llm"] = {"docs": TRIO, "per_doc": [{"doc": r["doc"], "cov": round(r["cov"], 3),
        "kept": r["n_kept"], "candidates": r["n_candidates"], "error": r["error"]} for r in rows],
        "mean_cov": round(mean_cov, 3) if ok else None, "bar_cov": 0.89, "cost": "~1x (single adjudication pass)",
        "pass": h260_pass, "verdict_recommendation": "GO" if h260_pass else "CLOSE"}
    log(f"H260 mean_cov={mean_cov:.3f} pass={h260_pass}")
except Exception as e:
    RESULTS["H260_llm"] = {"error": f"{type(e).__name__}: {str(e)[:200]}", "verdict_recommendation": "ERROR"}
    print("H260 ERROR:", RESULTS["H260_llm"]["error"])

  3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1 kept=90 cov=0.97 (cands=110, err=None)


  BMC_RESmart_AutoCPAP_User_Manual.pdf         kept=62 cov=0.76 (cands=100, err=None)


  Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf  kept=23 cov=1.00 (cands=24, err=None)
H260 mean kept cov (3/3 ok)=0.91 (bar>=0.89) -> PASS


## Gate 8 - H261 few-shot union-set demos (1 target doc)

Single pass on a target doc primed with 2 union-set demonstrations from the frozen low-leakage demo docs
(each demo = document text + its full union-of-5 entity list). **Bar**: >= +20 pts over the target doc's
single-pass baseline.

In [14]:
try:
    def demo_block(doc):
        names = sorted({n for r in RUNS for n in A[r].get(doc, [])})
        txt = DOCTEXT[doc]
        if len(txt) > 4000: txt = txt[:4000]
        return (f"EXAMPLE DOCUMENT:\n{txt}\n\nEXHAUSTIVE ENTITY LIST FOR THE EXAMPLE:\n" +
                json.dumps({"entities": [{"name": n} for n in names]}))
    fewshot_sys = base_system(
        "\n\nTwo worked examples below show a document and its exhaustive entity list. Match that thoroughness "
        "on the target document.")
    fewshot_user = ("\n\n".join(demo_block(d) for d in DEMOS) +
                    "\n\n=== TARGET DOCUMENT (extract exhaustively) ===\n" + DOCTEXT[DOC_FEWSHOT])
    names, err = safe_names(fewshot_sys, fewshot_user, temperature=0.0)
    fs_cov = cov_of(names, DOC_FEWSHOT); bc = BASE[DOC_FEWSHOT]; lift = (fs_cov - bc) * 100
    h261_pass = (err is None) and lift >= 20
    print(f"H261 target={DOC_FEWSHOT} demos={DEMOS}")
    print(f"  few-shot cov={fs_cov:.2f} baseline={bc:.2f} lift={lift:+.1f} pts (bar>=20, err={err}) -> {'PASS' if h261_pass else 'FAIL'}")
    save("h261_fewshot.json", {"target": DOC_FEWSHOT, "demos": DEMOS, "names": names, "fewshot_cov": fs_cov, "base_cov": bc, "lift_pts": lift, "error": err})
    RESULTS["H261"] = {"target": DOC_FEWSHOT, "demos": DEMOS, "fewshot_cov": round(fs_cov, 3), "base_cov": round(bc, 3),
        "lift_pts": round(lift, 1), "bar_pts": 20, "error": err, "pass": h261_pass,
        "verdict_recommendation": "GO" if h261_pass else "CLOSE"}
    log(f"H261 cov={fs_cov:.3f} base={bc:.3f} lift={lift:.1f} pass={h261_pass}")
except Exception as e:
    RESULTS["H261"] = {"error": f"{type(e).__name__}: {str(e)[:200]}", "verdict_recommendation": "ERROR"}
    print("H261 ERROR:", RESULTS["H261"]["error"])

H261 target=Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf demos=['CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf', 'CPAP-Machines-Brochure.pdf']
  few-shot cov=0.17 baseline=0.22 lift=-4.3 pts (bar>=20, err=None) -> FAIL


## Report assembly

Per-gate metrics + GO/NO-GO recommendation, written to `reports/llm-gates-r24c-<UTCstamp>.json`.

In [15]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = {
    "round": "R23/R24", "tier": "LLM-gate batch (local vLLM gpt-oss-120b)",
    "executor": "Claude (opus executor)", "generated_utc": stamp,
    "endpoint": ENDPOINT, "model": MODEL, "purpose": PURPOSE,
    "gold_carrier_definition": "token_set_ratio(name.lower(), product.lower()) >= 85; "
        "per-doc reference = that doc's union-of-5 (u5doc); global union-of-5 = 63 carriers",
    "anchors": {"union5": U, "single_run_mean_cov": round(single_cov, 4)},
    "target_docs": {"enumerate": DOC_ENUM, "logprobs": DOC_LOGP, "nsampling": DOC_NSAMP,
                    "trio": TRIO, "fewshot_target": DOC_FEWSHOT, "fewshot_demos": DEMOS},
    "serving_note": "gpt-oss-120b on vLLM emits verbose harmony reasoning; batched n>1 requests throw an "
                    "intermittent 'Unexpected token 200005' harmony-parse 500 - all gate calls are wrapped so "
                    "one failure never aborts the batch (max_tokens caps bound runaway temp>0 generations)",
    "gates": RESULTS,
}
out = REPORTS_DIR / f"llm-gates-r24c-{stamp}.json"
out.write_text(json.dumps(report, indent=2))
print("report written:", out)
for h, r in RESULTS.items():
    print(f"  {h:10s} {r.get('verdict_recommendation','?'):6s}  pass={r.get('pass', r.get('gate_close'))}")
log(f"report written {out}")

report written: reports/llm-gates-r24c-20260708T111638Z.json
  H243       GO      pass=False
  H246       GO      pass=True
  H245       CLOSE   pass=False
  H257       CLOSE   pass=None
  H251       NO-GO   pass=False
  H248       CLOSE   pass=False
  H258_llm   CLOSE   pass=False
  H260_llm   GO      pass=True
  H261       CLOSE   pass=False
